# La legge dentro la loss

Il codice del capitolo [«La legge dentro la loss»](https://book.paithon.it/main/PINN/come-funziona.html) — *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro — [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q numpy torch torchvision

## La legge dentro la loss

[Leggi la pagina](https://book.paithon.it/main/PINN/come-funziona.html)


### La PINN, riga per riga


In [ ]:
import numpy as npimport torchfrom torch import nntorch.manual_seed(42)# Parametri fisici della molla: massa, smorzamento, rigidezzam, c, k = 1.0, 0.4, 4.0# La candidata soluzione: un MLP che da t produce u(t)rete = nn.Sequential(    nn.Linear(1, 32), nn.Tanh(),    nn.Linear(32, 32), nn.Tanh(),    nn.Linear(32, 32), nn.Tanh(),    nn.Linear(32, 1),)# Punti di collocazione: 200 istanti a caso in [0, 10]t_c = 10.0 * torch.rand(200, 1)     # shape (200, 1)t_c.requires_grad_(True)            # derivate RISPETTO ALL'INPUT# L'istante iniziale, dove imporremo u(0)=1 e u'(0)=0t_0 = torch.zeros(1, 1, requires_grad=True)ottimizzatore = torch.optim.Adam(rete.parameters(), lr=1e-3)

### L'istante iniziale, dove imporremo u(0)=1 e u'(0)=0


In [ ]:
for epoca in range(30_000):    ottimizzatore.zero_grad()    # 1) fisica: residuo m*u'' + c*u' + k*u sui punti di collocazione    u = rete(t_c)                                        # shape (200, 1)    u_t = torch.autograd.grad(u, t_c, torch.ones_like(u),                              create_graph=True)[0]      # u'(t)    u_tt = torch.autograd.grad(u_t, t_c, torch.ones_like(u_t),                               create_graph=True)[0]     # u''(t)    residuo = m * u_tt + c * u_t + k * u    loss_fisica = (residuo ** 2).mean()    # 2) condizioni iniziali: u(0) = 1 e u'(0) = 0    u_0 = rete(t_0)    u_t0 = torch.autograd.grad(u_0, t_0, torch.ones_like(u_0),                               create_graph=True)[0]    loss_iniziale = (u_0 - 1.0).pow(2).mean() + u_t0.pow(2).mean()    # 3) loss totale, con piu' peso all'unico ancoraggio che abbiamo    loss = loss_fisica + 100.0 * loss_iniziale    loss.backward()    ottimizzatore.step()    if epoca % 5_000 == 0:        print(f"epoca {epoca:6d} | loss {loss.item():.2e}")

In [ ]:
# La soluzione analitica, per dare i voti alla retegamma = c / (2 * m)                        # 0.2omega_d = np.sqrt(k / m - gamma ** 2)      # sqrt(3.96) ~ 1.98997t_test = np.linspace(0.0, 10.0, 500)u_esatta = np.exp(-gamma * t_test) * (    np.cos(omega_d * t_test) + (gamma / omega_d) * np.sin(omega_d * t_test))with torch.no_grad():   # solo valutazione: registratore spento    t_torch = torch.tensor(t_test, dtype=torch.float32).reshape(-1, 1)    u_pinn = rete(t_torch).squeeze().numpy()print(f"errore massimo: {np.abs(u_pinn - u_esatta).max():.1e}")

### Il problema inverso, in tre righe di codice


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

```


## Dove la fisica aiuta e dove no

[Leggi la pagina](https://book.paithon.it/main/PINN/applicazioni-limiti.html)


### Il problema inverso, cioè il superpotere


In [ ]:
import torch# Il parametro fisico ignoto (qui la diffusività) diventa una manopola# addestrabile, indistinguibile da un peso qualsiasi della rete.alpha = torch.nn.Parameter(torch.tensor(0.5))          # valore iniziale di comodoottimizzatore = torch.optim.Adam(                      # ottimizzato insieme ai pesi    list(rete.parameters()) + [alpha], lr=1e-3)